# Estadística con Seaborn

Notebook práctico de la sección **4.3** del manual de Seaborn. Incluye gráficos de distribución, categóricos y matriciales; grids; y regresión lineal.

**Cómo usarlo:** ejecuta las celdas en orden. Los datos son sintéticos y se generan aquí mismo, por lo que no hace falta descargar archivos. Si falta alguna biblioteca, instala `seaborn`, `pandas`, `numpy` y `matplotlib` en el entorno del notebook.

## 1. Preparación e inspección de los datos

Seaborn trabaja de forma natural con DataFrames en formato largo: una fila por observación y una columna por variable. La semilla aleatoria hace reproducible el ejemplo.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
print("Seaborn:", sns.__version__)

In [ ]:
rng = np.random.default_rng(42)
n = 180
grupo = np.repeat(["Control", "Tratamiento A", "Tratamiento B"], n // 3)
horas = rng.uniform(0, 10, n)
efecto_grupo = pd.Series(grupo).map({
    "Control": 0,
    "Tratamiento A": 4,
    "Tratamiento B": 8,
}).to_numpy()
respuesta = 12 + 2.5 * horas + efecto_grupo + rng.normal(0, 5, n)

df = pd.DataFrame({
    "grupo": grupo,
    "horas": horas,
    "respuesta": respuesta,
    "medicion": rng.normal(50, 12, n),
})

display(df.head())
display(df.groupby("grupo").agg(
    n=("respuesta", "size"),
    media_respuesta=("respuesta", "mean"),
    desviacion=("respuesta", "std"),
).round(2))

## 2. Gráficos de distribución

`histplot` muestra frecuencias por intervalos, `kdeplot` estima una densidad suavizada y `ecdfplot` presenta la proporción acumulada sin definir intervalos.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(data=df, x="respuesta", bins=18, ax=axes[0])
axes[0].set(title="Histograma", xlabel="Respuesta")

sns.kdeplot(data=df, x="respuesta", fill=True, ax=axes[1])
axes[1].set(title="Densidad estimada", xlabel="Respuesta")

sns.ecdfplot(data=df, x="respuesta", ax=axes[2])
axes[2].set(title="Distribución acumulada", xlabel="Respuesta", ylabel="Proporción")

fig.tight_layout()
plt.show()

También se pueden comparar grupos con `hue`. `common_norm=False` normaliza cada curva por separado, de modo que se compara la forma y no el tamaño del grupo. La densidad es una estimación suavizada, no los datos observados.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(
    data=df, x="respuesta", hue="grupo", bins=16,
    element="step", stat="density", common_norm=False, ax=axes[0]
)
axes[0].set_title("Histograma normalizado por grupo")

sns.kdeplot(
    data=df, x="respuesta", hue="grupo", fill=True,
    common_norm=False, alpha=0.25, ax=axes[1]
)
axes[1].set_title("Densidad por grupo")

fig.tight_layout()
plt.show()

## 3. Gráficos categóricos

Los puntos muestran observaciones individuales; el boxplot resume mediana y cuartiles; el violín añade una densidad suavizada. Conviene mostrar puntos o distribuciones cuando una media por sí sola ocultaría la variabilidad.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.stripplot(data=df, x="grupo", y="respuesta", jitter=True,
              alpha=0.55, ax=axes[0])
axes[0].set_title("Observaciones")

sns.boxplot(data=df, x="grupo", y="respuesta", ax=axes[1])
axes[1].set_title("Mediana y cuartiles")

sns.violinplot(data=df, x="grupo", y="respuesta",
               inner="quart", cut=0, ax=axes[2])
axes[2].set_title("Densidad y cuartiles")

for ax in axes:
    ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="grupo", ax=axes[0])
axes[0].set(title="Número de observaciones", xlabel="Grupo", ylabel="Conteo")

sns.barplot(data=df, x="grupo", y="respuesta", ci=95, ax=axes[1])
axes[1].set(title="Media e intervalo de confianza", xlabel="Grupo",
            ylabel="Respuesta promedio")

fig.tight_layout()
plt.show()

`countplot` cuenta filas por categoría. `barplot` resume una variable numérica y dibuja por defecto un intervalo de confianza alrededor de la media; ese intervalo no es el rango de los datos.

## 4. Gráficos matriciales

Un heatmap usa color para representar valores de una matriz. Aquí se visualiza la correlación entre las variables numéricas. La correlación describe asociación lineal; no implica causalidad.

In [ ]:
variables = ["horas", "respuesta", "medicion"]
matriz_corr = df[variables].corr()

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(matriz_corr, annot=True, fmt=".2f", cmap="vlag",
            center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Correlación de Pearson")
plt.show()

In [ ]:
tabla_medias = df.pivot_table(
    index="grupo",
    values=variables,
    aggfunc="mean",
)

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(tabla_medias, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax)
ax.set_title("Promedio de variables por grupo")
plt.show()

In [ ]:
# Clustering exploratorio de variables según sus correlaciones.
sns.clustermap(matriz_corr, annot=True, fmt=".2f", cmap="vlag",
               center=0, figsize=(6, 6))
plt.show()

## 5. Grids y paneles

Las funciones de figura permiten repartir categorías en paneles. Usa `hue` para compararlas dentro del mismo eje y `col` o `row` para separarlas. Mantener escalas compartidas facilita la comparación.

In [ ]:
g = sns.relplot(
    data=df, x="horas", y="respuesta", hue="grupo",
    col="grupo", kind="scatter", height=3.5,
)
g.set_axis_labels("Horas", "Respuesta")
g.set_titles("Grupo: {col_name}")
plt.show()

In [ ]:
g = sns.displot(data=df, x="medicion", col="grupo", kind="hist",
                bins=15, height=3.5)
g.set_axis_labels("Medición", "Frecuencia")
plt.show()

In [ ]:
g = sns.catplot(data=df, x="grupo", y="respuesta", col="grupo",
                kind="violin", height=3.5, cut=0)
g.set_axis_labels("Grupo", "Respuesta")
plt.show()

`FacetGrid` ofrece control explícito: se define el grid y luego se mapea una función a cada panel.

In [ ]:
g = sns.FacetGrid(df, col="grupo", height=3.5,
                  sharex=True, sharey=True)
g.map_dataframe(sns.scatterplot, x="horas", y="respuesta", alpha=0.7)
g.set_axis_labels("Horas", "Respuesta")
g.set_titles("Grupo: {col_name}")
plt.show()

`pairplot` ofrece una vista rápida de pares de variables numéricas. `PairGrid` permite asignar gráficos distintos a la diagonal y a los paneles fuera de ella.

In [ ]:
sns.pairplot(data=df, vars=variables, hue="grupo", corner=True,
             diag_kind="hist", plot_kws={"alpha": 0.6})
plt.show()

In [ ]:
g = sns.PairGrid(df, vars=variables, hue="grupo", corner=True)
g.map_diag(sns.histplot, alpha=0.5)
g.map_offdiag(sns.scatterplot, alpha=0.6)
g.add_legend()
plt.show()

`jointplot` y `JointGrid` muestran la relación entre dos variables junto con sus distribuciones marginales. Limita el número de variables en grids grandes para mantenerlos legibles.

In [ ]:
sns.jointplot(data=df, x="horas", y="respuesta", kind="scatter",
              height=6, joint_kws={"alpha": 0.6})
plt.show()

In [ ]:
g = sns.JointGrid(data=df, x="horas", y="respuesta", height=6)
g.plot_joint(sns.scatterplot, alpha=0.6)
g.plot_marginals(sns.histplot, kde=True)
plt.show()

## 6. Gráficos de regresión lineal

`regplot` dibuja observaciones y una tendencia estimada en un solo eje. `lmplot` facilita estimar líneas por grupo (`hue`) o en paneles (`col`). Una asociación gráfica no demuestra causalidad.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.regplot(data=df, x="horas", y="respuesta", ci=95,
            scatter_kws={"alpha": 0.5},
            line_kws={"color": "black"}, ax=ax)
ax.set(title="Regresión lineal", xlabel="Horas", ylabel="Respuesta")
plt.show()

Se puede usar `regplot` también con vectores directos, sin pasar un DataFrame ni nombres de columna. `ci=None` oculta la banda de incertidumbre.

In [ ]:
sns.regplot(x=df["horas"], y=df["respuesta"], ci=None,
            scatter_kws={"alpha": 0.4})
plt.xlabel("Horas")
plt.ylabel("Respuesta")
plt.title("Regresión usando series de pandas")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.regplot(data=df, x="horas", y="respuesta", order=2,
            ci=None, scatter_kws={"alpha": 0.4}, ax=axes[0])
axes[0].set_title("Ajuste polinómico (order=2)")

sns.regplot(data=df, x="horas", y="respuesta", robust=True,
            ci=None, scatter_kws={"alpha": 0.4}, ax=axes[1])
axes[1].set_title("Ajuste robusto")

sns.regplot(data=df, x="horas", y="respuesta", lowess=True,
            ci=None, scatter_kws={"alpha": 0.4}, ax=axes[2])
axes[2].set_title("Tendencia local (LOWESS)")

fig.tight_layout()
plt.show()

`order=2` modela una curva polinómica; `robust=True` reduce la influencia de valores extremos; `lowess=True` dibuja una tendencia local que no es una recta lineal. Estas opciones no sustituyen la revisión de supuestos ni la validación del modelo.

In [ ]:
sns.lmplot(data=df, x="horas", y="respuesta", hue="grupo",
           height=5, ci=95,
           scatter_kws={"alpha": 0.5})
plt.show()

In [ ]:
sns.lmplot(data=df, x="horas", y="respuesta", col="grupo",
           col_wrap=3, height=3.5, ci=95,
           scatter_kws={"alpha": 0.5})
plt.show()

## 7. Resumen y práctica

- Distribuciones: `histplot`, `kdeplot`, `ecdfplot`.
- Comparaciones categóricas: `stripplot`, `boxplot`, `violinplot`, `countplot`, `barplot`.
- Matrices: `heatmap` y `clustermap`.
- Grids: `relplot`, `displot`, `catplot`, `FacetGrid`, `PairGrid`, `JointGrid`.
- Regresión: `regplot` para un eje; `lmplot` para grupos y facetas.

**Ejercicios:** cambia la variable de respuesta por `medicion`; compara densidades con histogramas; modifica el tipo de `kind` en `catplot` y `jointplot`; y compara las regresiones con `hue` frente a `col`. Describe qué se observa y evita afirmar causalidad.